# Mission 2 - Fusion(4번) 담당자용 Audio 예측 결과 export

4번(Audio+ASR Text Fusion) 담당 팀원이 요청한 형식대로, 우리 팀 최종 제출
모델(실험2가 대회 규정 위반(컨텍스트 마진)으로 폐기되어, 재실험한 버전 -
`Submission/Mission2/`에 있는 것과 동일한 구성: `N_UNFROZEN_LAYERS=4`,
컨텍스트 마진 없음, 사전 추출 segment 기준)로 **모든 발화 조각**(train
23,360통화 + val 5,840통화, metadata.csv 기준 segment 889,358개 /
chunk_manifest.csv 기준 청크 898,526개)에 대해
`speaker=1(신고자)`일 확률을 계산해서 csv로 저장한다.

- `04_train_wav2vec2_colab.ipynb`가 이미 Drive에 올려둔 shard(OGG로
  트랜스코딩된 segment 오디오) + `chunk_manifest.csv`를 그대로
  재사용한다 (재다운로드/재패킹 없음).
- 그 학습이 저장한 체크포인트(`mission2_wav2vec2_best.pt`)를 그대로
  로드한다 (재학습 없음).
- **출력**: `audio_train_predictions.csv`, `audio_val_predictions.csv`
  (컬럼: `call_id,segment_id,speaker,audio_prob`, `audio_prob` =
  `speaker=1(신고자)`일 확률). Fusion 요청 조건대로 train/val을 따로
  저장하고, 식별 기준은 `call_id + segment_id`, 확률 방향은 항상
  `speaker=1` 기준으로 통일했다.
- 덤으로: `audio_prob`을 만드는 김에 나오는 분류기 직전 pooled
  feature(분류기 head 바로 앞 벡터, Fusion 요청의 "가능하면 추가"에
  해당)도 `audio_train_embeddings.npy` / `audio_val_embeddings.npy`
  (+ 행 순서를 밝혀주는 `_index.csv`)로 같이 저장한다. 확률을 계산하는
  forward pass에서 그대로 얻어지는 값이라 추가 계산 비용은 거의 없다
  (저장 용량만 늘어남 - float16로 저장해 최소화함). 필요 없으면 아래
  `SAVE_EMBEDDINGS = False`로 끄면 된다.

**실행 순서**: `DRY_RUN = True`인 상태로 먼저 끝까지 한 번 돌려서(일부
청크만 처리 - 수 분 내 끝남) 에러 없이 csv/npy가 나오는지 확인한 뒤,
`DRY_RUN = False`로 바꿔서 본 실행을 돌릴 것. 본 실행은 gradient 계산이
없어 학습보다는 훨씬 빠르지만, 898,526개 청크 전체를 처리하므로 정확한
소요 시간은 실측이 필요하다 (GPU 런타임 권장 - 런타임 유형을 T4 GPU로
설정할 것).

## 0. 환경 설정

In [ ]:
!pip install -q transformers torchaudio soundfile

In [ ]:
# 실험1 재실험 학습(04_train_wav2vec2_colab.ipynb)과 같은 Drive 폴더를
# 그대로 재사용한다 - shard/manifest 재업로드 없음, 체크포인트도 그때
# 저장된 것을 그대로 읽는다.
DRIVE_ROOT = "/content/drive/MyDrive/dcc_mission2"
DRIVE_UPLOAD_DIR = f"{DRIVE_ROOT}/upload"
DRIVE_CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
DRIVE_EXPORT_DIR = f"{DRIVE_ROOT}/fusion_export"

PRETRAINED_CKPT = "Kkonjeong/wav2vec2-base-korean"
CHECKPOINT_NAME = "mission2_wav2vec2_best.pt"
CKPT_PATH = f"{DRIVE_CKPT_DIR}/{CHECKPOINT_NAME}"

LOCAL_DATA_DIR = "/content/data"
LOCAL_WAV_DIR = f"{LOCAL_DATA_DIR}/wav"
MANIFEST_PATH = f"{LOCAL_DATA_DIR}/chunk_manifest.csv"

OUTPUT_DIR = "/content/fusion_export"

# 파이프라인만 먼저 검증하고 싶으면 True (일부 청크만 처리, 수 분 내 완료).
# 실제 제출용 파일을 만들 때는 반드시 False로 바꿔서 전체를 돌릴 것.
DRY_RUN = True
DRY_RUN_LIMIT = 300

# 분류기 직전 pooled feature(임베딩)도 같이 저장할지 여부.
# 확률 계산과 같은 forward pass에서 나오는 값이라 켜둬도 계산 비용은
# 거의 늘지 않는다(저장 용량만 증가).
SAVE_EMBEDDINGS = True

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1. 데이터 다운로드

`03_pack_for_colab.py`가 만들어서 이미 Drive에 올려둔
`shards/shard_*.zip`(발화 segment 단위 OGG 압축, 889,358개 세그먼트
전체) + `manifest/chunk_manifest.csv`를 그대로 받는다. train/val 모두
같은 shard 안에 섞여 있으므로 이 하나로 충분하다 (split은 manifest의
`split` 컬럼으로 구분).

In [ ]:
import os
import shutil
import zipfile
import glob

os.makedirs(LOCAL_WAV_DIR, exist_ok=True)

if len(os.listdir(LOCAL_WAV_DIR)) == 0:

    def copy_and_unzip(zip_path, dest_dir):
        local_zip = f"/content/_tmp_{os.path.basename(zip_path)}"
        shutil.copy2(zip_path, local_zip)
        with zipfile.ZipFile(local_zip, "r") as zf:
            zf.extractall(dest_dir)
        os.remove(local_zip)

    shard_zips = sorted(glob.glob(f"{DRIVE_UPLOAD_DIR}/shards/shard_*.zip"))
    print("shard 개수:", len(shard_zips))
    for i, zip_path in enumerate(shard_zips, start=1):
        print(f"[{i}/{len(shard_zips)}] {zip_path}")
        copy_and_unzip(zip_path, LOCAL_WAV_DIR)

    shutil.copy2(
        f"{DRIVE_UPLOAD_DIR}/manifest/chunk_manifest.csv",
        MANIFEST_PATH
    )

else:
    print("이미 로컬에 데이터가 있음 - 다운로드/압축해제 건너뜀")

print("wav(실제로는 ogg) 파일 수:", len(os.listdir(LOCAL_WAV_DIR)))

In [ ]:
shutil.copy2(
    f"{DRIVE_UPLOAD_DIR}/wav2vec2_common.py",
    "/content/wav2vec2_common.py"
)

import sys
sys.path.append("/content")

from wav2vec2_common import TARGET_SR, MIN_DUR_SEC

print("TARGET_SR:", TARGET_SR, "/ MIN_DUR_SEC:", MIN_DUR_SEC)

## 2. Dataset

학습 때 쓴 `SpeakerSegmentChunkDataset`(04_train_wav2vec2_colab.ipynb)과
거의 같지만 두 가지가 다르다:

- `split`으로 필터링하지 않는다 (train/val 전부 처리한 뒤, 저장 단계에서
  `split` 컬럼으로 나눈다).
- `chunk_manifest.csv`에는 `segment_id` 컬럼이 따로 없어서, `segment_file`
  이름(`{call_id}_seg{index:04d}.ogg`)에서 index를 파싱해 그대로 쓴다
  (`Common/metadata.csv`의 `segment_id`와 같은 값 - 세그먼트 추출 시
  발화 순서 index를 그대로 파일명에 넣어뒀기 때문).

In [ ]:
import csv
from pathlib import Path

import numpy as np
import torch
import torchaudio
from torch.utils.data import Dataset


def parse_segment_id(segment_file):
    # "{call_id}_seg{index:04d}.ogg"(또는 .wav) -> index
    stem = Path(segment_file).stem
    return int(stem.rsplit("_seg", 1)[1])


class FusionExportDataset(Dataset):

    def __init__(self, manifest_path, wav_dir):
        self.wav_dir = wav_dir

        available_files = set(os.listdir(wav_dir))

        self.rows = []
        skipped = 0

        with open(manifest_path, "r", encoding="utf-8-sig", newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row["segment_file"] not in available_files:
                    skipped += 1
                    continue
                self.rows.append(row)

        print(f"대상 청크: {len(self.rows)}개, 로컬에 없어서 건너뜀: {skipped}개")

        if DRY_RUN:
            stride = max(1, len(self.rows) // DRY_RUN_LIMIT)
            self.rows = self.rows[::stride][:DRY_RUN_LIMIT]
            print(f"[DRY_RUN] {len(self.rows)}개 청크만 처리")

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]

        # segment_file 자체가 이미 해당 발화 구간만 잘라낸 wav(exp1
        # 파이프라인은 컨텍스트 마진 없이 startAt/endAt 그대로 크롭함).
        # chunk_start_sec/chunk_end_sec은 그 segment 파일 안에서의
        # 오프셋(대부분 [0, 전체길이] - 8초 넘는 소수 segment만 슬라이딩
        # window로 여러 청크로 나뉨). 학습 때(SpeakerSegmentChunkDataset)와
        # 완전히 동일한 로직이라야 train/inference mismatch가 없다.
        wav_path = f"{self.wav_dir}/{row['segment_file']}"
        waveform, orig_sr = torchaudio.load(wav_path)

        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        start = int(float(row["chunk_start_sec"]) * orig_sr)
        end = int(float(row["chunk_end_sec"]) * orig_sr)
        waveform = waveform[:, start:end]

        if orig_sr != TARGET_SR:
            waveform = torchaudio.functional.resample(waveform, orig_sr, TARGET_SR)

        min_samples = int(MIN_DUR_SEC * TARGET_SR)
        if waveform.shape[1] < min_samples:
            pad = min_samples - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, pad))

        return {
            "input_values": waveform.squeeze(0).numpy(),
            "call_id": row["call_id"],
            "segment_id": parse_segment_id(row["segment_file"]),
            "speaker": int(row["speaker"]),
            "split": row["split"]
        }

## 3. Collate + DataLoader

In [ ]:
from transformers import Wav2Vec2FeatureExtractor
from torch.utils.data import DataLoader

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(PRETRAINED_CKPT)


def collate_fn(batch):
    input_values = [b["input_values"] for b in batch]

    features = feature_extractor(
        input_values,
        sampling_rate=TARGET_SR,
        padding=True,
        return_tensors="pt"
    )

    return {
        "input_values": features["input_values"],
        "attention_mask": features.get("attention_mask"),
        "call_ids": [b["call_id"] for b in batch],
        "segment_ids": [b["segment_id"] for b in batch],
        "speakers": [b["speaker"] for b in batch],
        "splits": [b["split"] for b in batch]
    }


dataset = FusionExportDataset(MANIFEST_PATH, LOCAL_WAV_DIR)

# 추론 전용(gradient 없음)이라 학습 때(16)보다 배치를 키워도 된다
BATCH_SIZE = 8 if DRY_RUN else 32
NUM_WORKERS = 2 if DRY_RUN else 4

loader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=NUM_WORKERS
)

print("총 배치 수:", len(loader))

## 4. 체크포인트 로드

In [ ]:
from transformers import AutoConfig, Wav2Vec2ForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

checkpoint = torch.load(CKPT_PATH, map_location=device)

assert checkpoint["pretrained_ckpt"] == PRETRAINED_CKPT, (
    f"체크포인트의 pretrained_ckpt({checkpoint['pretrained_ckpt']})가 "
    f"위에서 설정한 PRETRAINED_CKPT({PRETRAINED_CKPT})와 다름"
)

config = AutoConfig.from_pretrained(
    checkpoint["pretrained_ckpt"], num_labels=checkpoint["num_labels"]
)
model = Wav2Vec2ForSequenceClassification(config)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

# 아래 추론 셀은 Wav2Vec2ForSequenceClassification.forward()의 pooling
# 로직을 use_weighted_layer_sum=False 기준으로 그대로 복제한 것 - 이
# 체크포인트가 그 경우가 아니면 아래 셀도 같이 고쳐야 한다.
assert not config.use_weighted_layer_sum, (
    "use_weighted_layer_sum=True인 체크포인트 - 아래 pooling 코드를 "
    "이 경우에 맞게 수정해야 함"
)

print("체크포인트 val_segment_level_acc (팀 자체 val split 기준):",
      checkpoint.get("val_segment_level_acc"))

## 5. 추론 - segment 단위 확률 + pooled feature 집계

한 발화 조각(segment)이 여러 청크로 쪼개진 경우, 청크별 확률/임베딩을
전부 모았다가 평균 낸다 (확률은 `evaluate()`/`inference.py`와 동일한
집계 방식, 임베딩도 같은 기준으로 맞춤).

확률만 필요하면 `model(input_values=..., attention_mask=...)`을 그냥
호출해도 되지만, pooled feature(임베딩)까지 같이 뽑으려고
`Wav2Vec2ForSequenceClassification.forward()` 내부 로직
(`wav2vec2` -> `projector` -> masked mean pooling -> `classifier`)을
그대로 풀어서 썼다 - 그래야 `model(...)`을 호출했을 때와 로짓이
완전히 동일하면서 pooled_output도 같이 얻을 수 있다.

In [ ]:
import time
from collections import defaultdict

segment_probs = defaultdict(list)
segment_embeds = defaultdict(list) if SAVE_EMBEDDINGS else None
segment_speaker = {}
segment_split = {}

t0 = time.time()

with torch.no_grad():
    for step, batch in enumerate(loader, start=1):
        input_values = batch["input_values"].to(device)

        attention_mask = batch["attention_mask"]
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)

        encoder_out = model.wav2vec2(
            input_values=input_values, attention_mask=attention_mask
        )
        hidden_states = model.projector(encoder_out.last_hidden_state)

        if attention_mask is None:
            pooled = hidden_states.mean(dim=1)
        else:
            padding_mask = model._get_feature_vector_attention_mask(
                hidden_states.shape[1], attention_mask
            )
            expand_mask = padding_mask.unsqueeze(-1).repeat(1, 1, hidden_states.shape[2])
            masked = hidden_states.clone()
            masked[~expand_mask] = 0.0
            pooled = masked.sum(dim=1) / padding_mask.sum(dim=1).view(-1, 1)

        logits = model.classifier(pooled)
        probs = torch.softmax(logits, dim=-1).cpu().numpy()

        if SAVE_EMBEDDINGS:
            pooled_np = pooled.cpu().numpy().astype(np.float16)

        for i in range(len(batch["call_ids"])):
            seg_key = (batch["call_ids"][i], batch["segment_ids"][i])
            segment_probs[seg_key].append(probs[i])
            segment_speaker[seg_key] = batch["speakers"][i]
            segment_split[seg_key] = batch["splits"][i]
            if SAVE_EMBEDDINGS:
                segment_embeds[seg_key].append(pooled_np[i])

        if step % 200 == 0 or step == len(loader):
            elapsed = time.time() - t0
            print(f"배치 {step}/{len(loader)} 처리 완료 ({elapsed / 60:.1f}분 경과)")

print("추론 완료. 발화 조각(segment) 수:", len(segment_probs))

## 6. csv / embedding 저장 (train / val 분리)

`audio_prob`은 반드시 `speaker=1(신고자)`일 확률이어야 한다는 게
Fusion 요청의 핵심 조건이므로, `mean_prob[1]`(index 1 = speaker 1)을
그대로 쓴다 - 학습 때 `label = int(row["speaker"])`로 라벨을 그대로
정수로 썼으므로(`04_train_wav2vec2_colab.ipynb`), 모델 logits의
index 1이 곧 speaker=1이다.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

rows_by_split = {"train": [], "val": []}
embed_rows_by_split = {"train": [], "val": []} if SAVE_EMBEDDINGS else None

unknown_split_count = 0

for seg_key, probs in segment_probs.items():
    call_id, segment_id = seg_key

    mean_prob = np.mean(probs, axis=0)
    audio_prob = float(mean_prob[1])  # speaker=1(신고자)일 확률

    speaker = segment_speaker[seg_key]
    split = segment_split[seg_key]

    if split not in rows_by_split:
        unknown_split_count += 1
        continue

    rows_by_split[split].append((call_id, segment_id, speaker, audio_prob))

    if SAVE_EMBEDDINGS:
        mean_embed = np.mean(segment_embeds[seg_key], axis=0)
        embed_rows_by_split[split].append((call_id, segment_id, mean_embed))

if unknown_split_count:
    print(f"[경고] split 값을 알 수 없어 건너뛴 segment: {unknown_split_count}개")

for split, out_name in [
    ("train", "audio_train_predictions.csv"),
    ("val", "audio_val_predictions.csv")
]:
    rows = sorted(rows_by_split[split], key=lambda r: (r[0], r[1]))
    out_path = f"{OUTPUT_DIR}/{out_name}"

    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["call_id", "segment_id", "speaker", "audio_prob"])
        writer.writerows(rows)

    print(f"{out_path} 저장 완료 ({len(rows)}행)")

if SAVE_EMBEDDINGS:
    for split, out_prefix in [
        ("train", "audio_train_embeddings"),
        ("val", "audio_val_embeddings")
    ]:
        items = sorted(embed_rows_by_split[split], key=lambda r: (r[0], r[1]))

        npy_path = f"{OUTPUT_DIR}/{out_prefix}.npy"
        idx_path = f"{OUTPUT_DIR}/{out_prefix}_index.csv"

        if items:
            matrix = np.stack([emb for _, _, emb in items]).astype(np.float16)
        else:
            matrix = np.zeros((0, 0), dtype=np.float16)

        np.save(npy_path, matrix)

        with open(idx_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["row_index", "call_id", "segment_id"])
            for i, (call_id, segment_id, _) in enumerate(items):
                writer.writerow([i, call_id, segment_id])

        print(f"{npy_path} 저장 완료 (shape={matrix.shape}, dtype=float16)")
        print(f"{idx_path} 저장 완료 - npy의 i번째 행 = 이 csv의 i번째 (call_id, segment_id)")

print()
print("생성된 파일:", sorted(os.listdir(OUTPUT_DIR)))

In [ ]:
# Colab 런타임이 끊겨도 결과가 남도록 Drive에도 백업
os.makedirs(DRIVE_EXPORT_DIR, exist_ok=True)
for fname in os.listdir(OUTPUT_DIR):
    shutil.copy2(f"{OUTPUT_DIR}/{fname}", f"{DRIVE_EXPORT_DIR}/{fname}")

print("Drive 백업 완료:", DRIVE_EXPORT_DIR)

## 마지막

1. `DRY_RUN = True`로 위 셀들을 전부 실행해서 에러 없이
   `audio_train_predictions.csv` 등이 만들어지는지 먼저 확인한다.
2. `DRY_RUN = False`로 바꾸고 런타임을 GPU로 확인한 뒤 처음부터 다시
   실행한다 (Dataset이 새로 만들어지도록 "5. 추론" 셀 이전부터 다시
   실행 - `FusionExportDataset`이 이미 `DRY_RUN` 결과로 만들어져
   있으면 안 됨).
3. `/content/fusion_export/`(+ Drive의 `dcc_mission2/fusion_export/`)에
   생긴 파일을 로컬 `Mission2/Work/Wav2Vec2/fusion_export/`로
   받는다.
4. `audio_train_predictions.csv`, `audio_val_predictions.csv`
   (+ 있으면 `audio_*_embeddings.npy`/`_index.csv`)를 4번(Fusion)
   담당자에게 **재전달**한다 - 이전에 보냈던 실험2(컨텍스트 마진 포함)
   버전은 대회 규정 위반으로 폐기됐으니 이번 파일로 교체해달라고
   같이 안내할 것.
5. 간단 검산: 각 csv에서 `speaker=1`인 행들의 `audio_prob` 평균이
   `speaker=0`인 행들보다 뚜렷하게 높아야 정상이다 (체크포인트
   공식 Validation segment-level accuracy가 0.9175이므로 당연히
   그래야 함) - 방향이 뒤집혀 있으면(즉 0이 더 높으면) `mean_prob`의
   index를 잘못 잡은 것이니 바로 잡을 것.